<a href="https://colab.research.google.com/github/rudalshan0412-code/attention-is-all-you-need-pytorch/blob/main/13)_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Drive 연결

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 프로젝트 경로 설정

from pathlib import Path
import sys


PROJECT_ROOT = Path(
    "/content/drive/MyDrive/attention_is_all_you_need"
)

SRC_DIR = PROJECT_ROOT / "src"

PROJECT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SRC_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

PROJECT_ROOT: /content/drive/MyDrive/attention_is_all_you_need
SRC_DIR: /content/drive/MyDrive/attention_is_all_you_need/src


In [ ]:
# 현재 src 파일 확인

for path in sorted(SRC_DIR.glob("*.py")):
    print(path.name)

attention.py
dataset.py
decoder.py
decoder_layer.py
encoder.py
encoder_layer.py
feed_forward.py
mask.py
multi_head_attention.py
positional_encoding.py
tokenizer.py
transformer.py
vocabulary.py


In [ ]:
# 필요한 모듈 import

import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from src.transformer import Transformer

from src.mask import (
    create_padding_mask,
    create_causal_mask,
)

from src.tokenizer import tokenize

from src.vocabulary import (
    Vocabulary,
    PAD_IDX,
    UNK_IDX,
    BOS_IDX,
    EOS_IDX,
)

from src.dataset import (
    TranslationDataset,
    collate_fn,
)

# 고정 인덱스 확인

print("PAD_IDX:", PAD_IDX)
print("UNK_IDX:", UNK_IDX)
print("BOS_IDX:", BOS_IDX)
print("EOS_IDX:", EOS_IDX)

PAD_IDX: 0
UNK_IDX: 1
BOS_IDX: 2
EOS_IDX: 3


In [ ]:
# Toy 데이터 준비

source_sentences = [
    "I love cats.",
    "You like dogs.",
    "We study attention.",
    "Transformers are powerful.",
    "I like apples.",
    "You love music.",
    "We study transformers.",
    "Cats are cute.",
]

target_sentences = [
    "Cats are lovely.",
    "Dogs are friendly.",
    "We learn attention.",
    "Transformers process sequences.",
    "Apples are delicious.",
    "Music is wonderful.",
    "We learn transformers.",
    "Cats are adorable.",
]

In [ ]:
# Tokenize

tokenized_source_sentences = [
    tokenize(sentence)
    for sentence in source_sentences
]

tokenized_target_sentences = [
    tokenize(sentence)
    for sentence in target_sentences
]


print(tokenized_source_sentences[0])
print(tokenized_target_sentences[0])

['i', 'love', 'cats', '.']
['cats', 'are', 'lovely', '.']


In [ ]:
# Source / Target Vocabulary 생성

source_vocab = Vocabulary(
    tokenized_source_sentences
)

target_vocab = Vocabulary(
    tokenized_target_sentences
)


print("Source Vocabulary Size:", len(source_vocab))
print("Target Vocabulary Size:", len(target_vocab))

print(
    "source_vocab is target_vocab:",
    source_vocab is target_vocab,
)

Source Vocabulary Size: 20
Target Vocabulary Size: 22
source_vocab is target_vocab: False


In [ ]:
# Dataset 생성

dataset = TranslationDataset(
    source_sentences,
    target_sentences,
    tokenize,
    tokenize,
    source_vocab,
    target_vocab,
)

# sample 확인

source_ids, target_ids = dataset[0]

print("source_ids:", source_ids)
print("target_ids:", target_ids)

print("source shape:", source_ids.shape)
print("target shape:", target_ids.shape)

source_ids: tensor([2, 4, 5, 6, 7, 3])
target_ids: tensor([2, 4, 5, 6, 7, 3])
source shape: torch.Size([6])
target shape: torch.Size([6])


In [ ]:
# DataLoader 생성

dataloader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn,
)

# 확인

source_batch, target_batch = next(
    iter(dataloader)
)

print("source_batch shape:", source_batch.shape)
print("target_batch shape:", target_batch.shape)

source_batch shape: torch.Size([8, 6])
target_batch shape: torch.Size([8, 6])


In [ ]:
# Device 설정

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("device:", device)

device: cuda


In [ ]:
# Transformer 생성

torch.manual_seed(42)

transformer = Transformer(
    source_vocab_size=len(source_vocab),
    target_vocab_size=len(target_vocab),
    d_model=32,
    num_heads=4,
    d_ff=64,
    num_encoder_layers=2,
    num_decoder_layers=2,
    max_len=50,
    dropout=0.1,
)

transformer = transformer.to(device)

In [ ]:
# Criterion / Optimizer

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)

optimizer = torch.optim.Adam(
    transformer.parameters(),
    lr=1e-3,
)

In [ ]:
# Toy model 간단 재학습

num_epochs = 100

loss_history = []

transformer.train()

for epoch in range(num_epochs):

    epoch_loss = 0.0

    for source_batch, target_batch in dataloader:

        source_batch = source_batch.to(device)
        target_batch = target_batch.to(device)

        decoder_input = target_batch[:, :-1]
        labels = target_batch[:, 1:]

        source_mask = create_padding_mask(
            source_batch,
            PAD_IDX,
        )

        target_padding_mask = create_padding_mask(
            decoder_input,
            PAD_IDX,
        )

        causal_mask = create_causal_mask(
            decoder_input.size(1),
            device=decoder_input.device,
        )

        target_mask = (
            target_padding_mask
            & causal_mask
        )

        optimizer.zero_grad()

        logits, _, _, _ = transformer(
            source_batch,
            decoder_input,
            source_mask,
            target_mask,
        )

        loss = criterion(
            logits.reshape(
                -1,
                logits.size(-1),
            ),
            labels.reshape(-1),
        )

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    epoch_loss /= len(dataloader)

    loss_history.append(epoch_loss)

    if (
        epoch == 0
        or (epoch + 1) % 20 == 0
    ):
        print(
            f"Epoch {epoch + 1:3d} "
            f"| Loss: {epoch_loss:.4f}"
        )

Epoch   1 | Loss: 3.1517
Epoch  20 | Loss: 1.8994
Epoch  40 | Loss: 1.1189
Epoch  60 | Loss: 0.6757
Epoch  80 | Loss: 0.4345
Epoch 100 | Loss: 0.2527


In [ ]:
# Loss 확인

print(
    "First Loss:",
    loss_history[0],
)

print(
    "Last Loss:",
    loss_history[-1],
)

First Loss: 3.151690721511841
Last Loss: 0.25267329812049866


In [ ]:
# model.eval()

transformer.eval()
# .train()을 사용하지 않고 .eval()을 사용하는 이유: Dropout을 끄기 위해서(좀 더 일정한 infernece 결과를 얻을 수 있다.)

Transformer(
  (source_embedding): Embedding(20, 32)
  (target_embedding): Embedding(22, 32)
  (source_positional_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (target_positional_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): Encoder(
    (layers): ModuleList(
      (0-1): 2 x EncoderLayer(
        (self_attention): MultiHeadAttention(
          (W_Q): Linear(in_features=32, out_features=32, bias=True)
          (W_K): Linear(in_features=32, out_features=32, bias=True)
          (W_V): Linear(in_features=32, out_features=32, bias=True)
          (attention): ScaledDotProductAttention()
          (W_O): Linear(in_features=32, out_features=32, bias=True)
        )
        (feed_forward): PositionwiseFeedForward(
          (linear1): Linear(in_features=32, out_features=64, bias=True)
          (linear2): Linear(in_features=64, out_features=32, bias=True)
        )
        (dropout1): Dropout(p=0.1, inplace=F

In [ ]:
# inference 할 문장 준비

source_sentence= "I love cats"

In [ ]:
# Source Tokenize

source_tokens = tokenize(
    source_sentence
)

print(source_tokens)

['i', 'love', 'cats']


In [ ]:
# Source Token IDs 생성

# Dataset을 사용하지 않기에 직접 BOS, EOS를 붙인다
source_token_ids = [
    BOS_IDX, # 고정 인덱스 2
    *source_vocab.encode(source_tokens),
    EOS_IDX, # 고정 인덱스 3
]

print(source_token_ids)

[2, 4, 5, 6, 3]


In [ ]:
# source_tensor

source_tensor = torch.tensor(
    [source_token_ids],
    dtype=torch.long,
    device=device,
)

print(source_tensor)
print(source_tensor.shape)
print(source_tensor.dtype)

tensor([[2, 4, 5, 6, 3]], device='cuda:0')
torch.Size([1, 5])
torch.int64


In [ ]:
# Source Mask 생성

source_mask = create_padding_mask(
    source_tensor,
    PAD_IDX,
)

print(source_mask)
print(source_mask.shape)

# 현재 PAD가 없음애도 mask를 만드는 이유: Transformer API와 동일한 방식으로 처리하기 위해서이다.

tensor([[[[True, True, True, True, True]]]], device='cuda:0')
torch.Size([1, 1, 1, 5])


In [ ]:
# Decoder를 BOS 하나로 시작

generated_ids = torch.tensor(
    [[BOS_IDX]],
    dtype=torch.long,
    device=device,
)

print(generated_ids)
print(generated_ids.shape)

# 현재 decoder가 알고 있는 것은 BOS 하나 뿐이다.

tensor([[2]], device='cuda:0')
torch.Size([1, 1])


In [ ]:
# 첫 Step Target Padding Mask

target_padding_mask = create_padding_mask(
    generated_ids, # 현재는 BOS
    PAD_IDX,
)

print(target_padding_mask)
print(target_padding_mask.shape) # (B, 1, 1, K)

tensor([[[[True]]]], device='cuda:0')
torch.Size([1, 1, 1, 1])


In [ ]:
# 첫 Step Causal Mask

causal_mask = create_causal_mask(
    generated_ids.size(1),
    device=generated_ids.device,
)

print(causal_mask)
print(causal_mask.shape) (1, 1, S, S)

# 이때, inference에는 미래 Token이 존재하지 않기에 causal mask가 필요없어보인다.
# 하지만, Training과 동일한 Decoder self-attention 구조를 그대로 사용하기에 규칙을 유지하기 위해 사용한다.

tensor([[[[True]]]], device='cuda:0')
torch.Size([1, 1, 1, 1])


In [ ]:
# Target Mask 결합

target_mask = (
    target_padding_mask
    & causal_mask
)

print(target_mask)
print(target_mask.shape)

tensor([[[[True]]]], device='cuda:0')
torch.Size([1, 1, 1, 1])


In [ ]:
# torch.no_grad()로 첫 Forward

with torch.no_grad():

    logits, _, _, _ = transformer(
        source_tensor,
        generated_ids,
        source_mask,
        target_mask,
    )

print("logits shape:", logits.shape) # (B, target_len, target_vacab_size)

logits shape: torch.Size([1, 1, 22])


In [ ]:
# 마지막 위치 Logits만 선택

next_token_logits = logits[
    :,
    -1,# 단순 정수임으로 차원 소멸
    :
]

print(
    "next_token_logits shape:",
    next_token_logits.shape,
)

next_token_logits shape: torch.Size([1, 22])


In [ ]:
# Greedy Argmax


# Greedy Argmax: 점수표에서 가장 가능성 높은 하나만을 선택해 인덱스 -> 토큰 과정을 진행한다.
# 이때, 어차피 가장 점수가 높은 값만이 필요하기에 softmax를 따로 진행하지 않는다(가장 큰 값은 softmax 를 진행해도 여전히 가장 크다).
next_token_id = torch.argmax(
    next_token_logits,
    dim=-1,
)

print(next_token_id)
print(next_token_id.shape)

tensor([4], device='cuda:0')
torch.Size([1])


In [ ]:
# 실제 예측 Token 확인

next_token = target_vocab.id_to_token(
    next_token_id.item()
)

print("next token id:", next_token_id.item())
print("next token:", next_token)

next token id: 4
next token: cats


In [ ]:
# unsqueeze 확인

next_token_id_column = (
    next_token_id.unsqueeze(1) # 현재 next_token_id는 (1,)이다. generated_ids와 shape((1, 1))를 맞춰주기 위해 차원 하나를 추가한다.
)

print(
    next_token_id_column.shape
)

torch.Size([1, 1])


In [ ]:
# torch.cat으로 뒤에 붙이기

generated_ids = torch.cat( # 뒤에 붙여줌으로서 <BOS> cats가 된다.
    [
        generated_ids,
        next_token_id.unsqueeze(1),
    ],
    dim=1,
)

print(generated_ids)
print(generated_ids.shape)

tensor([[2, 4]], device='cuda:0')
torch.Size([1, 2])


In [ ]:
# 다시 mask 제작

# 현재 generated_ids가 (1, 2)이기에 다시 mask를 만든다
# target의 length가 계속해서 하나씩 증가하기에 mask도 계속해서 커져야한다.

target_padding_mask = create_padding_mask(
    generated_ids,
    PAD_IDX,
)

causal_mask = create_causal_mask(
    generated_ids.size(1),
    device=generated_ids.device,
)

target_mask = (
    target_padding_mask
    & causal_mask
)


print(
    "target_padding_mask:",
    target_padding_mask.shape,
)

print(
    "causal_mask:",
    causal_mask.shape,
)

print(
    "target_mask:",
    target_mask.shape,
)

target_padding_mask: torch.Size([1, 1, 1, 2])
causal_mask: torch.Size([1, 1, 2, 2])
target_mask: torch.Size([1, 1, 2, 2])


In [ ]:
# 두번째 Forward Shape 확인

with torch.no_grad():

    logits, _, _, _ = transformer(
        source_tensor,
        generated_ids,
        source_mask,
        target_mask,
    )


print(logits.shape) # target의 길이가 1 늘었음을 알 수 있다.

torch.Size([1, 2, 22])


In [ ]:
# Greedy Decoding Loop

generated_ids = torch.tensor( # 기존 테스트에서 token 하나를 append 했기에, 다시 BOS 하나부터 초기화한다.
    [[BOS_IDX]],
    dtype=torch.long,
    device=device,
)

max_generation_length = 20 # 만약 잘못 학습되어 EOS를 생성하지 못할 경우를 대비하여 최대 루프를 지정한다.

with torch.no_grad():

    for step in range(
        max_generation_length
    ):

        print(
            f"\nStep {step + 1}"
        )

        print(
            "Current generated_ids shape:",
            generated_ids.shape,
        )

        target_padding_mask = (
            create_padding_mask(
                generated_ids,
                PAD_IDX,
            )
        )

        causal_mask = (
            create_causal_mask(
                generated_ids.size(1),
                device=generated_ids.device,
            )
        )

        target_mask = (
            target_padding_mask
            & causal_mask
        )

        logits, _, _, _ = transformer(
            source_tensor,
            generated_ids,
            source_mask,
            target_mask,
        )

        print(
            "Logits shape:",
            logits.shape,
        )

        next_token_logits = logits[
            :,
            -1,
            :
        ]

        next_token_id = torch.argmax(
            next_token_logits,
            dim=-1,
        )

        next_token = (
            target_vocab.id_to_token(
                next_token_id.item()
            )
        )

        print(
            "Predicted token:",
            next_token,
        )

        generated_ids = torch.cat(
            [
                generated_ids,
                next_token_id.unsqueeze(1),
            ],
            dim=1,
        )

        print(
            "After append:",
            generated_ids.shape,
        )

        if ( # EOS 가 나타난 순간 종료
            next_token_id.item()
            == EOS_IDX
        ):
            print(
                "<EOS> generated. Stop."
            )
            break


Step 1
Current generated_ids shape: torch.Size([1, 1])
Logits shape: torch.Size([1, 1, 22])
Predicted token: cats
After append: torch.Size([1, 2])

Step 2
Current generated_ids shape: torch.Size([1, 2])
Logits shape: torch.Size([1, 2, 22])
Predicted token: are
After append: torch.Size([1, 3])

Step 3
Current generated_ids shape: torch.Size([1, 3])
Logits shape: torch.Size([1, 3, 22])
Predicted token: lovely
After append: torch.Size([1, 4])

Step 4
Current generated_ids shape: torch.Size([1, 4])
Logits shape: torch.Size([1, 4, 22])
Predicted token: .
After append: torch.Size([1, 5])

Step 5
Current generated_ids shape: torch.Size([1, 5])
Logits shape: torch.Size([1, 5, 22])
Predicted token: <EOS>
After append: torch.Size([1, 6])
<EOS> generated. Stop.


In [ ]:
# 최종 generated_ids 확인

print(
    "Final generated_ids:",
    generated_ids,
)

print(
    "Final shape:",
    generated_ids.shape,
)

Final generated_ids: tensor([[2, 4, 5, 6, 7, 3]], device='cuda:0')
Final shape: torch.Size([1, 6])


In [ ]:
# Token으로 Decode

generated_id_list = (
    generated_ids
    .squeeze(0)
    .tolist()
)

generated_tokens_all = (
    target_vocab.decode(
        generated_id_list
    )
)

print(
    "All tokens:",
    generated_tokens_all,
)

All tokens: ['<BOS>', 'cats', 'are', 'lovely', '.', '<EOS>']


In [ ]:
# BOS, EOS, PAD 제거

# UNK는 제거하지 않는다.

filtered_ids = [
    token_id
    for token_id in generated_id_list
    if token_id not in {
        PAD_IDX,
        BOS_IDX,
        EOS_IDX,
    }
]

generated_tokens = (
    target_vocab.decode(
        filtered_ids
    )
)

print(
    "Generated tokens:",
    generated_tokens,
)

Generated tokens: ['cats', 'are', 'lovely', '.']


In [ ]:
# 문자열로 출력

generated_sentence = " ".join(
    generated_tokens
)

print(
    "Generated sentence:",
    generated_sentence,
)
# 마침표 앞에 공백이 생길 수 있다.

Generated sentence: cats are lovely .


In [ ]:
# 다른 Training 문장으로 한번 더 테스트

source_sentence = (
    "You love music."
)

In [ ]:
# greedy_decode() 함수

def greedy_decode(
    transformer,
    source_sentence,
    source_vocab,
    target_vocab,
    source_tokenizer,
    device,
    max_generation_length=20,
):

    transformer.eval()

    source_tokens = source_tokenizer(
        source_sentence
    )

    source_token_ids = [
        BOS_IDX,
        *source_vocab.encode(
            source_tokens
        ),
        EOS_IDX,
    ]

    source_tensor = torch.tensor(
        [source_token_ids],
        dtype=torch.long,
        device=device,
    )

    source_mask = create_padding_mask(
        source_tensor,
        PAD_IDX,
    )

    generated_ids = torch.tensor(
        [[BOS_IDX]],
        dtype=torch.long,
        device=device,
    )

    with torch.no_grad():

        for _ in range(
            max_generation_length
        ):

            target_padding_mask = (
                create_padding_mask(
                    generated_ids,
                    PAD_IDX,
                )
            )

            causal_mask = (
                create_causal_mask(
                    generated_ids.size(1),
                    device=generated_ids.device,
                )
            )

            target_mask = (
                target_padding_mask
                & causal_mask
            )

            logits, _, _, _ = transformer(
                source_tensor,
                generated_ids,
                source_mask,
                target_mask,
            )

            next_token_logits = logits[
                :,
                -1,
                :
            ]

            next_token_id = torch.argmax(
                next_token_logits,
                dim=-1,
            )

            generated_ids = torch.cat(
                [
                    generated_ids,
                    next_token_id.unsqueeze(1),
                ],
                dim=1,
            )

            if (
                next_token_id.item()
                == EOS_IDX
            ):
                break

    generated_id_list = (
        generated_ids
        .squeeze(0)
        .tolist()
    )

    filtered_ids = [
        token_id
        for token_id in generated_id_list
        if token_id not in {
            PAD_IDX,
            BOS_IDX,
            EOS_IDX,
        }
    ]

    generated_tokens = (
        target_vocab.decode(
            filtered_ids
        )
    )

    generated_sentence = " ".join(
        generated_tokens
    )

    generated_sentence = (
        generated_sentence
        .replace(" .", ".")
        .replace(" ,", ",")
        .replace(" !", "!")
        .replace(" ?", "?")
    )

    return (
        generated_tokens,
        generated_sentence,
    )

In [ ]:
# 함수 테스트1

tokens, sentence = greedy_decode(
    transformer=transformer,
    source_sentence="I love cats.",
    source_vocab=source_vocab,
    target_vocab=target_vocab,
    source_tokenizer=tokenize,
    device=device,
)

print("Tokens:", tokens)
print("Sentence:", sentence)

Tokens: ['cats', 'are', 'lovely', '.']
Sentence: cats are lovely.


In [ ]:
# 함수 테스트 2

tokens, sentence = greedy_decode(
    transformer=transformer,
    source_sentence="You love music.",
    source_vocab=source_vocab,
    target_vocab=target_vocab,
    source_tokenizer=tokenize,
    device=device,
)

print("Tokens:", tokens)
print("Sentence:", sentence)

Tokens: ['music', 'is', 'wonderful', '.']
Sentence: music is wonderful.


In [ ]:
# 최종 통합 테스트

assert PAD_IDX == 0
assert UNK_IDX == 1
assert BOS_IDX == 2
assert EOS_IDX == 3

assert source_vocab is not target_vocab


test_sentence = "I love cats."

test_tokens = tokenize(
    test_sentence
)

test_source_ids = [
    BOS_IDX,
    *source_vocab.encode(test_tokens),
    EOS_IDX,
]

test_source_tensor = torch.tensor(
    [test_source_ids],
    dtype=torch.long,
    device=device,
)

assert (
    test_source_tensor.dtype
    == torch.long
)

assert (
    test_source_tensor.ndim
    == 2
)

assert (
    test_source_tensor.shape[0]
    == 1
)

assert (
    test_source_tensor[0, 0].item()
    == BOS_IDX
)

assert (
    test_source_tensor[0, -1].item()
    == EOS_IDX
)


test_source_mask = (
    create_padding_mask(
        test_source_tensor,
        PAD_IDX,
    )
)

assert (
    test_source_mask.shape
    ==
    (
        1,
        1,
        1,
        test_source_tensor.size(1),
    )
)


test_generated_ids = torch.tensor(
    [[BOS_IDX]],
    dtype=torch.long,
    device=device,
)

assert (
    test_generated_ids.shape
    == (1, 1)
)

assert (
    test_generated_ids[0, 0].item()
    == BOS_IDX
)


test_target_padding_mask = (
    create_padding_mask(
        test_generated_ids,
        PAD_IDX,
    )
)

test_causal_mask = (
    create_causal_mask(
        test_generated_ids.size(1),
        device=test_generated_ids.device,
    )
)

test_target_mask = (
    test_target_padding_mask
    & test_causal_mask
)


transformer.eval()

with torch.no_grad():

    test_logits, _, _, _ = transformer(
        test_source_tensor,
        test_generated_ids,
        test_source_mask,
        test_target_mask,
    )


assert (
    test_logits.shape
    ==
    (
        1,
        1,
        len(target_vocab),
    )
)


test_next_token_logits = (
    test_logits[:, -1, :]
)

assert (
    test_next_token_logits.shape
    ==
    (
        1,
        len(target_vocab),
    )
)


test_next_token_id = torch.argmax(
    test_next_token_logits,
    dim=-1,
)

assert (
    test_next_token_id.shape
    == (1,)
)


old_length = (
    test_generated_ids.size(1)
)

test_generated_ids = torch.cat(
    [
        test_generated_ids,
        test_next_token_id.unsqueeze(1),
    ],
    dim=1,
)

assert (
    test_generated_ids.size(1)
    ==
    old_length + 1
)

assert (
    test_generated_ids.dtype
    == torch.long
)

assert (
    test_generated_ids[0, 0].item()
    == BOS_IDX
)


print(
    "Inference basic integration test passed."
)

Inference basic integration test passed.


In [ ]:
# 전체 Greedy Decoding 통합 확인

test_tokens, test_sentence = (
    greedy_decode(
        transformer=transformer,
        source_sentence="I love cats.",
        source_vocab=source_vocab,
        target_vocab=target_vocab,
        source_tokenizer=tokenize,
        device=device,
        max_generation_length=20,
    )
)

print(
    "Generated Tokens:",
    test_tokens,
)

print(
    "Generated Sentence:",
    test_sentence,
)

Generated Tokens: ['cats', 'are', 'lovely', '.']
Generated Sentence: cats are lovely.
